# T41 / E17 — Chuyển miền sang tin tức: ViFactCheck, split gốc

Cùng công thức với E15 (ViWikiFC), chỉ đổi bộ dữ liệu. Ba bộ đã chạy là hỏi đáp (ViHallu), văn
bản dài (ISE-DSC01) và bách khoa (ViWikiFC); ViFactCheck là **tin tức** — miền thứ ba. Câu hỏi
CH3: tín hiệu chú ý có khái quát hóa sang miền này không.

**Quyết định 22/10:** chạy trước, xem kết quả rồi mới quyết định có đưa vào quyển không. Nếu chỉ
lặp lại kết luận của E15/E16, ghi một đoạn ngắn ở mục 8.2 thay vì thêm bảng.

| Việc | Thời gian |
|---|---|
| Chuẩn bị dữ liệu, kiểm thử | ~2 phút, CPU |
| Trích 7.232 mẫu (5.062 / 723 / 1.447), trung bình ~960 token | **~2 giờ** |
| Soi shard, chép kết quả | ~1 phút, CPU |

Hai cấu hình (chunk-aware và mốc lookback gộp) **dùng chung một lượt trích** vì trùng
`extraction_hash` (`cdffb5c66735`).

## Chấm điểm — KHÔNG chạy ở đây

Quy tắc chốt ở T23: mọi phép so sánh chấm trên **cùng một máy**, vì điểm dev lệch tới 0,0075
giữa Kaggle và máy cá nhân. Tải `ket_qua_t41/` về, đặt shard vào `data/processed/`, rồi trên máy:

```
python scripts/run_chunk_aware.py --config configs/e17_baseline_lookback_vifactcheck.yaml
python scripts/run_chunk_aware.py --config configs/e17_chunk_vifactcheck.yaml
```

In [1]:
# Ô 1 — lấy code. Chạy lại được nhiều lần.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    done = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(" ".join(args) + chr(10) + done.stdout + done.stderr)
    return done.stdout.strip()


if (REPO_DIR / ".git").is_dir():
    run("git", "fetch", "--quiet", "origin", cwd=REPO_DIR)
    run("git", "reset", "--quiet", "--hard", "origin/main", cwd=REPO_DIR)
    print("đã cập nhật repo có sẵn")
else:
    run("git", "clone", "--quiet", REPO_URL, str(REPO_DIR))
    print("đã clone mới")

%cd /kaggle/working/vihallulens
print("commit:", run("git", "log", "--oneline", "-1", cwd=REPO_DIR))

đã clone mới
/kaggle/working/vihallulens
commit: f0ad3c2 Merge pull request #134 from wsunicorn/t50-giai-thich-hoi-dap


In [2]:
# Ô 2 — cài đặt, ghim phiên bản theo uv.lock (requirements-kaggle.txt, từ 22/10/2026).
# `--no-deps` cố ý không kéo phụ thuộc nào về, nên rank-bm25 (BM25 dựng ngữ cảnh) nằm trong file ghim.
!pip install -q --no-deps -e .
!pip install -q -r requirements-kaggle.txt

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for vihallulens (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 108.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vihallulens 0.1.0 requires pyvi, which is not installed.


In [3]:
# Ô 3 — TIỀN KIỂM. Vài giây. Chặn những gì phiên này không tự tạo được.
import importlib.util
import sys
from pathlib import Path

import torch

sys.path.insert(0, "src")
from vihallulens.config import extraction_hash, load_config
from vihallulens.data.paths import find_raw_dir

problems = []
packages = ("torch", "transformers", "bitsandbytes", "pandas", "pyarrow")
absent = [name for name in packages if importlib.util.find_spec(name) is None]
print(f"  gói phải có sẵn   : {'THIẾU ' + str(absent) if absent else 'đủ'}")
if absent:
    problems.append(f"thiếu gói {absent}")
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ"
print(f"  GPU               : {gpu}")
if not torch.cuda.is_available():
    problems.append("chưa bật GPU T4")
try:
    raw = Path(find_raw_dir())
    files = sorted(p.name for p in raw.glob("vifactcheck_*.parquet"))
    print(f"  dữ liệu thô       : {raw} — {files or 'KHÔNG CÓ vifactcheck'}")
    if len(files) < 3:
        problems.append(f"cần ba file vifactcheck_train/dev/test, chỉ thấy {files}")
except Exception as error:
    print(f"  dữ liệu thô       : KHÔNG TÌM THẤY ({error})")
    problems.append("chưa gắn dataset dữ liệu thô")
hashes = {extraction_hash(load_config(p)) for p in ("configs/e17_chunk_vifactcheck.yaml", "configs/e17_baseline_lookback_vifactcheck.yaml")}
print(f"  hash trích        : {sorted(hashes)} (phải đúng một giá trị)")
if len(hashes) != 1:
    problems.append("hai cấu hình không chung lượt trích")
if problems:
    raise SystemExit("TIỀN KIỂM HỎNG: " + "; ".join(problems))
print("Tiền kiểm đạt.")

  gói phải có sẵn   : đủ
  GPU               : Tesla T4
  dữ liệu thô       : /kaggle/input/datasets/unicorn1209/vihallulens — ['vifactcheck_dev.parquet', 'vifactcheck_test.parquet', 'vifactcheck_train.parquet']
  hash trích        : ['cdffb5c66735'] (phải đúng một giá trị)
Tiền kiểm đạt.


In [4]:
# Ô 4 — chuẩn bị dữ liệu và kiểm thử. Khoảng 2 phút, CPU.
# split_data GIỮ NGUYÊN split gốc cho vifactcheck (KEEP_ORIGINAL trong scripts/split_data.py).
!python scripts/probe_env.py
!python scripts/normalize_data.py --dataset vifactcheck
!python scripts/split_data.py --only vifactcheck
!python -m pytest tests/test_vifactcheck.py tests/test_attention_math.py tests/test_drop_nonfinite.py -q


MÔI TRƯỜNG
  repo             : /kaggle/working/vihallulens
  commit           : f0ad3c2 Merge pull request #134 from wsunicorn/t50-giai-thich-hoi-dap
  python           : 3.13.15
  torch            : 2.11.0+cu128
  transformers     : 5.17.0
  bitsandbytes     : 0.50.2
  accelerate       : 1.15.0
  vihallulens      : 0.1.0 tại /kaggle/working/vihallulens/src/vihallulens/__init__.py
  dữ liệu          : /kaggle/input/datasets/unicorn1209/vihallulens  (14 file)
      MANIFEST.md
      isedsc01_test_private.json
      isedsc01_test_public.json
      isedsc01_train.json
      vifactcheck_dataset_card.md
      vifactcheck_dev.parquet
      vifactcheck_gitattributes.txt
      vifactcheck_test.parquet
      vifactcheck_train.parquet
      vihallu_test_public.csv
      vihallu_train.csv
      viwikifc_dev.csv
      viwikifc_test.csv
      viwikifc_train.csv

CHUẨN HÓA VIFACTCHECK
  nguồn                 : /kaggle/input/datasets/unicorn1209/vihallulens
  số dòng               : 7,232
  ngữ cản

## Trích đặc trưng

Một ô cho cả ba tập. Hỏng tập nào thì ghi lại và đi tiếp, **không dừng** — trích có lưu tiến độ,
nên phần dở dang vẫn mang về được và phiên sau chạy tiếp từ chỗ đó (bài học T30, T31).

In [5]:
# Ô 5 — TRÍCH ĐẶC TRƯNG. Khoảng 2 giờ. Chạy lại được, có lưu tiến độ.
import os

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["PYTHONUNBUFFERED"] = "1"

failed = []
for split in ("train", "dev", "test"):
    print("=" * 78)
    print(f"  TRÍCH — vifactcheck/{split}")
    print("=" * 78)
    code = os.system(f"python scripts/extract_features.py --config configs/e17_chunk_vifactcheck.yaml --split {split}")
    if code:
        failed.append(f"{split} (mã lỗi {code})")
        print(f"  !! {split} hỏng, mã lỗi {code} — ghi lại và đi tiếp")
print(f"  TẬP HỎNG: {failed}" if failed else "  Cả ba tập trích xong.")

  TRÍCH — vifactcheck/train

T20 — TRÍCH ĐẶC TRƯNG LOOKBACK
  cấu hình              : configs/e17_chunk_vifactcheck.yaml  (hash cdffb5c66735)
  mô hình đọc           : Qwen/Qwen2.5-7B-Instruct
  lượng tử hóa / kiểu số: nf4 / float16
  bỏ lớp                : [27]
  trần token ngữ cảnh   : 4,096
  chia đoạn             : sentence
  bộ dữ liệu            : vifactcheck/train, 5,062 mẫu
  khối đặc trưng ghi ra : lookback_total, lookback_context, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  mẫu có bằng chứng     : 2,998/5,062  → ghi thêm gold_rank cho E06
  đã có sẵn             : 0 mẫu trong vifactcheck_train_cdffb5c66735.jsonl
  còn phải chạy         : 5,062 mẫu


       5062/5062  1,329 ms/mẫu  còn ~0 phút  lỗi 0

--------------------------------------------------------------------------------
  đã ghi thêm           : 5,062 mẫu, tổng 5,062
  lỗi                   : 0
  bị cắt ngữ cảnh       : 8/5,062
  có lớp tràn số        : 0/5,062
  thời gian             : 112.1 phút, 1,329 ms/mẫu
  file                  : data/processed/vifactcheck_train_cdffb5c66735.jsonl
  TRÍCH — vifactcheck/dev

T20 — TRÍCH ĐẶC TRƯNG LOOKBACK
  cấu hình              : configs/e17_chunk_vifactcheck.yaml  (hash cdffb5c66735)
  mô hình đọc           : Qwen/Qwen2.5-7B-Instruct
  lượng tử hóa / kiểu số: nf4 / float16
  bỏ lớp                : [27]
  trần token ngữ cảnh   : 4,096
  chia đoạn             : sentence
  bộ dữ liệu            : vifactcheck/dev, 723 mẫu
  khối đặc trưng ghi ra : lookback_total, lookback_context, chunk_entropy, chunk_max_share, chunk_gini, top1_top2_gap, chunk_drift
  mẫu có bằng chứng     : 430/723  → ghi thêm gold_rank cho E06
  đã có sẵn        

In [6]:
# Ô 6 — soi shard. Vài giây, CPU. Không dừng notebook dù shard có nan.
!python scripts/inspect_shard.py --config configs/e17_chunk_vifactcheck.yaml


SOI SHARD — e17_chunk_vifactcheck
  mô hình đọc       : Qwen/Qwen2.5-7B-Instruct
  đang bỏ lớp       : [27]
  hash trích        : cdffb5c66735

  TRAIN  5,062 mẫu   lưới 27 × 28
    mẫu có lớp tràn số : 0  (0.00 %)

  DEV  723 mẫu   lưới 27 × 28
    mẫu có lớp tràn số : 0  (0.00 %)

  TEST  1,447 mẫu   lưới 27 × 28
    mẫu có lớp tràn số : 0  (0.00 %)

------------------------------------------------------------------------------------
  Không lớp nào tràn số. Shard sạch.


In [7]:
# Ô 7 — LẤY KẾT QUẢ VỀ. Ô quan trọng nhất của phiên.
import shutil
from pathlib import Path

out = Path("/kaggle/working/ket_qua_t41")
out.mkdir(exist_ok=True)
run_hash = extraction_hash(load_config("configs/e17_chunk_vifactcheck.yaml"))
for split in ("train", "dev", "test"):
    src = Path(f"data/processed/vifactcheck_{split}_{run_hash}.jsonl")
    if src.exists():
        shutil.copy(src, out / src.name)
    else:
        print(f"  !! thiếu {src}")
for path in ("configs/e17_chunk_vifactcheck.yaml", "configs/e17_baseline_lookback_vifactcheck.yaml"):
    shutil.copy(path, out / Path(path).name)
for f in sorted(out.iterdir()):
    print(f"  {f.name:<46} {f.stat().st_size / 1e6:>8.1f} MB")
print("Tải cả thư mục ket_qua_t41/ và notebook đã chạy.")

  e17_baseline_lookback_vifactcheck.yaml              0.0 MB
  e17_chunk_vifactcheck.yaml                          0.0 MB
  vifactcheck_dev_cdffb5c66735.jsonl                 39.2 MB
  vifactcheck_test_cdffb5c66735.jsonl                78.5 MB
  vifactcheck_train_cdffb5c66735.jsonl              274.6 MB
Tải cả thư mục ket_qua_t41/ và notebook đã chạy.
